# ML Network Intrusion Detector
**IBM SkillsBuild Cyber Security Bootcamp 2026 — BharatCares**  
Author: Ashfaq

This notebook builds a Scikit-learn model that classifies network connections as **Normal** or one of three attack categories — **DoS** (denial of service), **Probe** (scanning/reconnaissance) and **R2L** (remote-to-local, e.g. password guessing) — using connection features modelled on the public NSL-KDD / KDD Cup 99 datasets.

**Pipeline:** Dataset → Preprocessing → EDA → Model Training → Evaluation → Detection of new connections

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_style("whitegrid")

## 1. Dataset
By default the notebook **generates a synthetic dataset** (6,000 connections) using the real KDD feature names and typical attack behaviour (e.g. SYN floods have many connections with a high SYN-error rate; port scans touch many different services and get rejected). A small share of attacks is made to look normal (stealthy attacks) and 2% of labels are flipped, so the task is realistic rather than trivial.

**Using a real dataset (recommended for final results):** download **NSL-KDD** (https://www.unb.ca/cic/datasets/nsl.html), map attack names to `Normal / DoS / Probe / R2L`, keep the columns listed in `FEATURES` below plus a `label` column, and save as `network_traffic.csv` next to this notebook. It will be loaded automatically.

In [ ]:
CATEGORICAL = ["protocol_type", "service", "flag"]
NUMERIC = ["duration", "src_bytes", "dst_bytes", "logged_in", "num_failed_logins", "count", "srv_count",
           "serror_rate", "rerror_rate", "same_srv_rate", "diff_srv_rate", "dst_host_count",
           "dst_host_srv_count", "dst_host_same_srv_rate"]
FEATURES = CATEGORICAL + NUMERIC

def clip01(a): return np.clip(a, 0, 1)

def make_class(label, n):
    d = {}
    if label == "Normal":
        d["protocol_type"] = rng.choice(["tcp", "udp", "icmp"], n, p=[0.8, 0.17, 0.03])
        d["service"] = rng.choice(["http", "smtp", "ftp", "dns", "ssh", "other"], n, p=[0.5, 0.12, 0.08, 0.15, 0.05, 0.1])
        d["flag"] = rng.choice(["SF", "REJ", "S0"], n, p=[0.94, 0.04, 0.02])
        d["duration"] = rng.exponential(5, n).round()
        d["src_bytes"] = rng.lognormal(6.5, 1.5, n).round()
        d["dst_bytes"] = rng.lognormal(7.5, 1.8, n).round()
        d["logged_in"] = rng.binomial(1, 0.7, n)
        d["num_failed_logins"] = rng.binomial(1, 0.02, n)
        d["count"] = rng.integers(1, 30, n)
        d["srv_count"] = rng.integers(1, 30, n)
        d["serror_rate"] = clip01(rng.normal(0.03, 0.05, n))
        d["rerror_rate"] = clip01(rng.normal(0.03, 0.05, n))
        d["same_srv_rate"] = clip01(rng.normal(0.9, 0.12, n))
        d["diff_srv_rate"] = clip01(rng.normal(0.05, 0.06, n))
        d["dst_host_count"] = rng.integers(1, 150, n)
        d["dst_host_srv_count"] = rng.integers(20, 255, n)
        d["dst_host_same_srv_rate"] = clip01(rng.normal(0.85, 0.15, n))
    elif label == "DoS":
        d["protocol_type"] = rng.choice(["tcp", "icmp", "udp"], n, p=[0.6, 0.35, 0.05])
        d["service"] = rng.choice(["http", "private", "ecr_i", "other"], n, p=[0.3, 0.3, 0.3, 0.1])
        d["flag"] = rng.choice(["S0", "SF", "REJ"], n, p=[0.8, 0.15, 0.05])
        d["duration"] = np.zeros(n)
        d["src_bytes"] = rng.choice([0, 1032, 520], n, p=[0.6, 0.3, 0.1]).astype(float)
        d["dst_bytes"] = np.zeros(n)
        d["logged_in"] = np.zeros(n, dtype=int)
        d["num_failed_logins"] = np.zeros(n, dtype=int)
        d["count"] = rng.integers(150, 511, n)
        d["srv_count"] = rng.integers(150, 511, n)
        d["serror_rate"] = clip01(rng.normal(0.9, 0.12, n))
        d["rerror_rate"] = clip01(rng.normal(0.02, 0.04, n))
        d["same_srv_rate"] = clip01(rng.normal(0.97, 0.05, n))
        d["diff_srv_rate"] = clip01(rng.normal(0.01, 0.02, n))
        d["dst_host_count"] = rng.integers(200, 256, n)
        d["dst_host_srv_count"] = rng.integers(10, 256, n)
        d["dst_host_same_srv_rate"] = clip01(rng.normal(0.9, 0.12, n))
    elif label == "Probe":
        d["protocol_type"] = rng.choice(["tcp", "icmp", "udp"], n, p=[0.7, 0.2, 0.1])
        d["service"] = rng.choice(["private", "other", "http", "ftp", "ssh", "smtp"], n)
        d["flag"] = rng.choice(["REJ", "SF", "S0", "RSTO"], n, p=[0.55, 0.2, 0.15, 0.1])
        d["duration"] = rng.exponential(1, n).round()
        d["src_bytes"] = rng.choice([0, 8, 18, 44], n).astype(float)
        d["dst_bytes"] = np.zeros(n)
        d["logged_in"] = np.zeros(n, dtype=int)
        d["num_failed_logins"] = np.zeros(n, dtype=int)
        d["count"] = rng.integers(5, 120, n)
        d["srv_count"] = rng.integers(1, 10, n)
        d["serror_rate"] = clip01(rng.normal(0.2, 0.2, n))
        d["rerror_rate"] = clip01(rng.normal(0.6, 0.25, n))
        d["same_srv_rate"] = clip01(rng.normal(0.2, 0.15, n))
        d["diff_srv_rate"] = clip01(rng.normal(0.6, 0.2, n))
        d["dst_host_count"] = rng.integers(100, 256, n)
        d["dst_host_srv_count"] = rng.integers(1, 40, n)
        d["dst_host_same_srv_rate"] = clip01(rng.normal(0.15, 0.12, n))
    else:  # R2L
        d["protocol_type"] = rng.choice(["tcp", "udp"], n, p=[0.95, 0.05])
        d["service"] = rng.choice(["ftp", "telnet", "ssh", "pop3", "smtp"], n, p=[0.35, 0.25, 0.15, 0.15, 0.1])
        d["flag"] = rng.choice(["SF", "RSTO", "REJ"], n, p=[0.85, 0.1, 0.05])
        d["duration"] = rng.exponential(40, n).round()
        d["src_bytes"] = rng.lognormal(5.5, 1.0, n).round()
        d["dst_bytes"] = rng.lognormal(6.0, 1.2, n).round()
        d["logged_in"] = rng.binomial(1, 0.3, n)
        d["num_failed_logins"] = rng.integers(1, 6, n)
        d["count"] = rng.integers(1, 15, n)
        d["srv_count"] = rng.integers(1, 15, n)
        d["serror_rate"] = clip01(rng.normal(0.04, 0.06, n))
        d["rerror_rate"] = clip01(rng.normal(0.1, 0.1, n))
        d["same_srv_rate"] = clip01(rng.normal(0.9, 0.12, n))
        d["diff_srv_rate"] = clip01(rng.normal(0.05, 0.06, n))
        d["dst_host_count"] = rng.integers(1, 80, n)
        d["dst_host_srv_count"] = rng.integers(1, 60, n)
        d["dst_host_same_srv_rate"] = clip01(rng.normal(0.8, 0.2, n))
    df = pd.DataFrame(d)
    df["label"] = label
    return df

def build_synthetic(n=6000, stealth=0.08, label_noise=0.02):
    sizes = {"Normal": 0.55, "DoS": 0.2, "Probe": 0.15, "R2L": 0.10}
    parts = []
    for lab, frac in sizes.items():
        part = make_class(lab, int(n * frac))
        if lab != "Normal":                         # stealthy attacks that look like normal traffic
            k = int(len(part) * stealth)
            idx = part.sample(k, random_state=SEED).index
            part.loc[idx, NUMERIC + CATEGORICAL] = make_class("Normal", k)[NUMERIC + CATEGORICAL].values
        parts.append(part)
    df = pd.concat(parts, ignore_index=True)
    flip = rng.random(len(df)) < label_noise
    df.loc[flip, "label"] = rng.choice(list(sizes), flip.sum())
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)

if os.path.exists("network_traffic.csv"):
    df = pd.read_csv("network_traffic.csv")[FEATURES + ["label"]].dropna()
    print("Loaded real dataset:", df.shape)
else:
    df = build_synthetic()
    print("Using synthetic dataset:", df.shape)
df.head()

In [ ]:
print(df["label"].value_counts())
df.groupby("label")[["count", "serror_rate", "rerror_rate", "diff_srv_rate", "num_failed_logins"]].mean().round(2)

## 2. Exploratory Data Analysis

In [ ]:
order = ["Normal", "DoS", "Probe", "R2L"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df["label"].value_counts().reindex(order).plot.bar(ax=axes[0], color=["#2E75B6", "#C0392B", "#E67E22", "#8E44AD"], rot=0)
axes[0].set_title("Traffic Class Distribution")
sns.boxplot(data=df, x="label", y="count", order=order, ax=axes[1], hue="label", legend=False)
axes[1].set_title("Connections to Same Host (count)")
sns.boxplot(data=df, x="label", y="rerror_rate", order=order, ax=axes[2], hue="label", legend=False)
axes[2].set_title("Rejection Error Rate")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
pd.crosstab(df["protocol_type"], df["label"], normalize="columns")[order].plot.bar(rot=0, ax=plt.gca())
plt.title("Protocol Mix per Traffic Class"); plt.ylabel("Share")
plt.tight_layout(); plt.show()

## 3. Preprocessing and Model Training
Categorical columns (protocol, service, flag) are one-hot encoded and numeric columns standardised inside a scikit-learn `Pipeline`, so exactly the same preprocessing is applied at prediction time. Data is split 80/20 (stratified). Three models are compared: Logistic Regression (baseline), Decision Tree and Random Forest.

In [ ]:
X, y = df[FEATURES], df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("num", StandardScaler(), NUMERIC),
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree":       DecisionTreeClassifier(max_depth=12, random_state=SEED),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
}

pipes, rows = {}, []
for name, clf in models.items():
    pipe = Pipeline([("prep", pre), ("clf", clf)]).fit(X_train, y_train)
    pipes[name] = pipe
    pred = pipe.predict(X_test)
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1_macro").mean()
    # binary view: did we correctly flag attack vs normal?
    attack_true, attack_pred = (y_test != "Normal"), (pred != "Normal")
    det_rate = (attack_true & attack_pred).sum() / attack_true.sum()
    fa_rate = (~attack_true & attack_pred).sum() / (~attack_true).sum()
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
                 "Macro F1": f1_score(y_test, pred, average="macro"),
                 "CV Macro F1": cv, "Attack Detection Rate": det_rate, "False Alarm Rate": fa_rate})

results = pd.DataFrame(rows).set_index("Model").round(4)
results

## 4. Evaluation of the Best Model

In [ ]:
best_name = results["Macro F1"].idxmax()
best = pipes[best_name]
pred = best.predict(X_test)
print("Best model:", best_name, "\n")
print(classification_report(y_test, pred, labels=order))

cm = confusion_matrix(y_test, pred, labels=order)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=order, yticklabels=order)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion Matrix – {best_name}")
plt.tight_layout(); plt.show()

In [ ]:
rf = pipes["Random Forest"]
names = rf.named_steps["prep"].get_feature_names_out()
imp = pd.Series(rf.named_steps["clf"].feature_importances_, index=names).sort_values()
imp.tail(12).plot.barh(figsize=(8, 5), color="#2E75B6")
plt.title("Top Feature Importances (Random Forest)"); plt.tight_layout(); plt.show()

## 5. Save the Model and Detect New Connections

In [ ]:
joblib.dump(best, "intrusion_model.joblib")

def detect(conn: dict):
    row = pd.DataFrame([conn])[FEATURES]
    proba = best.predict_proba(row)[0]
    k = proba.argmax()
    return best.classes_[k], round(float(proba[k]), 3)

normal_conn = dict(protocol_type="tcp", service="http", flag="SF", duration=3, src_bytes=520, dst_bytes=4300,
                   logged_in=1, num_failed_logins=0, count=4, srv_count=4, serror_rate=0.0, rerror_rate=0.0,
                   same_srv_rate=1.0, diff_srv_rate=0.0, dst_host_count=20, dst_host_srv_count=120, dst_host_same_srv_rate=0.9)
syn_flood = dict(protocol_type="tcp", service="private", flag="S0", duration=0, src_bytes=0, dst_bytes=0,
                 logged_in=0, num_failed_logins=0, count=480, srv_count=480, serror_rate=1.0, rerror_rate=0.0,
                 same_srv_rate=1.0, diff_srv_rate=0.0, dst_host_count=255, dst_host_srv_count=255, dst_host_same_srv_rate=1.0)
port_scan = dict(protocol_type="tcp", service="other", flag="REJ", duration=0, src_bytes=0, dst_bytes=0,
                 logged_in=0, num_failed_logins=0, count=90, srv_count=2, serror_rate=0.1, rerror_rate=0.8,
                 same_srv_rate=0.1, diff_srv_rate=0.8, dst_host_count=250, dst_host_srv_count=10, dst_host_same_srv_rate=0.05)
guess_pw = dict(protocol_type="tcp", service="ftp", flag="SF", duration=60, src_bytes=200, dst_bytes=300,
                logged_in=0, num_failed_logins=4, count=3, srv_count=3, serror_rate=0.0, rerror_rate=0.1,
                same_srv_rate=1.0, diff_srv_rate=0.0, dst_host_count=10, dst_host_srv_count=8, dst_host_same_srv_rate=0.9)

for name, conn in [("Normal web request", normal_conn), ("SYN flood", syn_flood),
                   ("Port scan", port_scan), ("FTP password guessing", guess_pw)]:
    label, p = detect(conn)
    print(f"{name:<24} -> {label:<7} (confidence {p})")

## 6. Conclusion & Limitations
* Connection-level statistics (connection counts, error rates, failed logins) separate the attack families well: precision is about 0.99 for every attack class and about 90% of attacks are detected with a near-zero false-alarm rate. The remaining misses are mostly the deliberately stealthy attacks that imitate normal traffic, which are classified as Normal.
* Tree-based models handle the mix of categorical and numeric features without heavy tuning and give interpretable feature importances.
* **Limitations:** the default dataset is synthetic, so results must be re-validated on NSL-KDD or CIC-IDS; the model is offline and does not handle new, unseen attack types or encrypted payload analysis.
* **Future work:** train on NSL-KDD / CIC-IDS2017, add an anomaly-detection model (Isolation Forest) for unknown attacks, and stream live traffic from Zeek/Suricata logs.